# Bouquet vs OMFIT: COCOS Conversion & Bt/Ip Flip Comparison

Field-by-field comparison of bouquet's `cocosify()` and `flip_Bt_Ip()`
against OMFIT's `OMFITgeqdsk.cocosify()` and `OMFITgeqdsk.flip_Bt_Ip()`.

---

**Requires `omfit_classes`.**  Install via:
```
pip install omfit_classes
```
This notebook will not run without it. The companion notebook
`cocos_and_save_example.ipynb` demonstrates bouquet's functionality
standalone.

---

**Test cases:**
1. COCOS 1 → 7 (sigma_Bp and sigma_rhotp sign flip)
2. COCOS 1 → 11 (2π factor on psi)
3. COCOS 7 → 1 (reverse of test 1)
4. COCOS 7 → 11
5. `flip_Bt_Ip` from COCOS 1
6. Full OMFIT workflow: load as COCOS 7, cocosify to 1, flip Bt/Ip

In [1]:
import sys, os
import numpy as np

# Ensure bouquet is importable from the repo root
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

try:
    from omfit_classes.omfit_eqdsk import OMFITgeqdsk
    print('omfit_classes loaded successfully')
except ImportError:
    raise ImportError(
        'This notebook requires omfit_classes.\n'
        'Install with: pip install omfit_classes\n'
        'See cocos_and_save_example.ipynb for the standalone bouquet demo.'
    )

from bouquet.io.geqdsk import GEQDSKEquilibrium, read_geqdsk
print('bouquet loaded successfully')

omfit_classes loaded successfully
bouquet loaded successfully


In [2]:
GFILE = '../D3D-like/TkMkr_D3Dlike_Hmode_BSamp=1.0.geqdsk'
assert os.path.exists(GFILE), f'{GFILE} not found — run from examples/COCOS_Bt_Ip/'

# Fields to compare (g-file key, friendly name)
FIELDS = [
    ('SIMAG',   'psi_axis'),
    ('SIBRY',   'psi_boundary'),
    ('BCENTR',  'B_center'),
    ('CURRENT', 'Ip'),
    ('FPOL',    'FPOL'),
    ('PRES',    'PRES'),
    ('PPRIME',  'PPRIME'),
    ('FFPRIM',  'FFPRIM'),
    ('QPSI',    'QPSI'),
    ('PSIRZ',   'PSIRZ'),
]

def compare(bouquet_raw, omfit_raw, label):
    """Compare every field between bouquet and OMFIT raw dicts."""
    print(f'\n{"=" * 60}')
    print(f'  {label}')
    print(f'{"=" * 60}')
    all_pass = True
    for gkey, name in FIELDS:
        bval = bouquet_raw[gkey]
        oval = omfit_raw[gkey]
        if isinstance(bval, np.ndarray):
            bval = bval.ravel()
            oval = np.asarray(oval).ravel()
            if len(bval) != len(oval):
                print(f'  {name:12s}  SIZE MISMATCH  bouquet={len(bval)}, omfit={len(oval)}')
                all_pass = False
                continue
            maxabs = max(np.max(np.abs(bval)), np.max(np.abs(oval)), 1e-30)
            maxdiff = np.max(np.abs(bval - oval))
            reldiff = maxdiff / maxabs
            ok = reldiff < 1e-10
            status = 'PASS' if ok else 'FAIL'
            print(f'  {name:12s}  max|diff|/max|val| = {reldiff:.2e}  [{status}]')
        else:
            bval = float(bval)
            oval = float(oval)
            denom = max(abs(bval), abs(oval), 1e-30)
            reldiff = abs(bval - oval) / denom
            ok = reldiff < 1e-10
            status = 'PASS' if ok else 'FAIL'
            print(f'  {name:12s}  bouquet={bval:>14.6e}  omfit={oval:>14.6e}  rel={reldiff:.2e}  [{status}]')
        if not ok:
            all_pass = False
    print(f'  {"—" * 40}')
    print(f'  Overall: {"ALL PASS" if all_pass else "SOME FAILURES"}')
    return all_pass

## Helper: run a COCOS conversion with both codes

In [3]:
def run_cocosify_comparison(cocos_in, cocos_out, label=None):
    """Cocosify the g-file from cocos_in → cocos_out with both bouquet and OMFIT."""
    if label is None:
        label = f'cocosify: COCOS {cocos_in} → {cocos_out}'

    # --- Bouquet ---
    beq = read_geqdsk(GFILE, cocos=cocos_in)
    beq.cocosify(cocos_out)

    # --- OMFIT ---
    oeq = OMFITgeqdsk(GFILE)
    oeq.load(raw=True)
    oeq._cocos = cocos_in
    oeq.cocosify(cocos_out, True, True)

    return compare(beq._raw, oeq, label)

## Test 1: COCOS 1 → 7

In [4]:
run_cocosify_comparison(1, 7);

Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...

  cocosify: COCOS 1 → 7
  psi_axis      bouquet= -3.482758e-01  omfit= -3.482758e-01  rel=0.00e+00  [PASS]
  psi_boundary  bouquet= -4.402488e-02  omfit= -4.402488e-02  rel=0.00e+00  [PASS]
  B_center      bouquet=  2.024348e+00  omfit=  2.024348e+00  rel=0.00e+00  [PASS]
  Ip            bouquet=  1.200040e+06  omfit=  1.200040e+06  rel=0.00e+00  [PASS]
  FPOL          max|diff|/max|val| = 0.00e+00  [PASS]
  PRES          max|diff|/max|val| = 0.00e+00  [PASS]
  PPRIME        max|diff|/max|val| = 0.00e+00  [PASS]
  FFPRIM        max|diff|/max|val| = 0.00e+00  [PASS]
  QPSI          max|diff|/max|val| = 0.00e+00  [PASS]
  PSIRZ         max|diff|/max|val| = 0.00e+00  [PASS]
  ————————————————————————————————————————
  Overall: ALL PASS


## Test 2: COCOS 1 → 11

In [5]:
run_cocosify_comparison(1, 11);

Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...

  cocosify: COCOS 1 → 11
  psi_axis      bouquet=  2.188282e+00  omfit=  2.188282e+00  rel=0.00e+00  [PASS]
  psi_boundary  bouquet=  2.766165e-01  omfit=  2.766165e-01  rel=0.00e+00  [PASS]
  B_center      bouquet=  2.024348e+00  omfit=  2.024348e+00  rel=0.00e+00  [PASS]
  Ip            bouquet=  1.200040e+06  omfit=  1.200040e+06  rel=0.00e+00  [PASS]
  FPOL          max|diff|/max|val| = 0.00e+00  [PASS]
  PRES          max|diff|/max|val| = 0.00e+00  [PASS]
  PPRIME        max|diff|/max|val| = 0.00e+00  [PASS]
  FFPRIM        max|diff|/max|val| = 0.00e+00  [PASS]
  QPSI          max|diff|/max|val| = 0.00e+00  [PASS]
  PSIRZ         max|diff|/max|val| = 0.00e+00  [PASS]
  ————————————————————————————————————————
  Overall: ALL PASS


## Test 3: COCOS 7 → 1

In [6]:
run_cocosify_comparison(7, 1);

Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...

  cocosify: COCOS 7 → 1
  psi_axis      bouquet= -3.482758e-01  omfit= -3.482758e-01  rel=0.00e+00  [PASS]
  psi_boundary  bouquet= -4.402488e-02  omfit= -4.402488e-02  rel=0.00e+00  [PASS]
  B_center      bouquet=  2.024348e+00  omfit=  2.024348e+00  rel=0.00e+00  [PASS]
  Ip            bouquet=  1.200040e+06  omfit=  1.200040e+06  rel=0.00e+00  [PASS]
  FPOL          max|diff|/max|val| = 0.00e+00  [PASS]
  PRES          max|diff|/max|val| = 0.00e+00  [PASS]
  PPRIME        max|diff|/max|val| = 0.00e+00  [PASS]
  FFPRIM        max|diff|/max|val| = 0.00e+00  [PASS]
  QPSI          max|diff|/max|val| = 0.00e+00  [PASS]
  PSIRZ         max|diff|/max|val| = 0.00e+00  [PASS]
  ————————————————————————————————————————
  Overall: ALL PASS


## Test 4: COCOS 7 → 11

In [7]:
run_cocosify_comparison(7, 11);

Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...

  cocosify: COCOS 7 → 11
  psi_axis      bouquet= -2.188282e+00  omfit= -2.188282e+00  rel=0.00e+00  [PASS]
  psi_boundary  bouquet= -2.766165e-01  omfit= -2.766165e-01  rel=0.00e+00  [PASS]
  B_center      bouquet=  2.024348e+00  omfit=  2.024348e+00  rel=0.00e+00  [PASS]
  Ip            bouquet=  1.200040e+06  omfit=  1.200040e+06  rel=0.00e+00  [PASS]
  FPOL          max|diff|/max|val| = 0.00e+00  [PASS]
  PRES          max|diff|/max|val| = 0.00e+00  [PASS]
  PPRIME        max|diff|/max|val| = 0.00e+00  [PASS]
  FFPRIM        max|diff|/max|val| = 0.00e+00  [PASS]
  QPSI          max|diff|/max|val| = 0.00e+00  [PASS]
  PSIRZ         max|diff|/max|val| = 0.00e+00  [PASS]
  ————————————————————————————————————————
  Overall: ALL PASS


## Test 5: flip_Bt_Ip from COCOS 1

In [8]:
# --- Bouquet ---
beq = read_geqdsk(GFILE, cocos=1)
beq.flip_Bt_Ip()

# --- OMFIT ---
oeq = OMFITgeqdsk(GFILE)
oeq.load(raw=True)
oeq._cocos = 1
oeq.cocosify(1, True, True)  # identity, but sets internal state
oeq.flip_Bt_Ip()

compare(beq._raw, oeq, 'flip_Bt_Ip (COCOS 1)');

Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...

  flip_Bt_Ip (COCOS 1)
  psi_axis      bouquet= -3.482758e-01  omfit= -3.482758e-01  rel=0.00e+00  [PASS]
  psi_boundary  bouquet= -4.402488e-02  omfit= -4.402488e-02  rel=0.00e+00  [PASS]
  B_center      bouquet= -2.024348e+00  omfit= -2.024348e+00  rel=0.00e+00  [PASS]
  Ip            bouquet= -1.200040e+06  omfit= -1.200040e+06  rel=0.00e+00  [PASS]
  FPOL          max|diff|/max|val| = 0.00e+00  [PASS]
  PRES          max|diff|/max|val| = 0.00e+00  [PASS]
  PPRIME        max|diff|/max|val| = 0.00e+00  [PASS]
  FFPRIM        max|diff|/max|val| = 0.00e+00  [PASS]
  QPSI          max|diff|/max|val| = 0.00e+00  [PASS]
  PSIRZ         max|diff|/max|val| = 0.00e+00  [PASS]
  ————————————————————————————————————————
  Overall: ALL PASS


## Test 6: Full OMFIT workflow — COCOS 7 → 1 then flip_Bt_Ip

Replicates:
```python
new_eq = OMFITgeqdsk(eqfile)
new_eq.load(raw=True)
new_eq._cocos = 7
new_eq.cocosify(1, True, True)
new_eq.flip_Bt_Ip()
```

In [9]:
# --- Bouquet ---
beq = read_geqdsk(GFILE, cocos=7)
beq.cocosify(1)
beq.flip_Bt_Ip()

# --- OMFIT ---
oeq = OMFITgeqdsk(GFILE)
oeq.load(raw=True)
oeq._cocos = 7
oeq.cocosify(1, True, True)
oeq.flip_Bt_Ip()

compare(beq._raw, oeq, 'Full workflow: COCOS 7 → 1 + flip_Bt_Ip');

Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...

  Full workflow: COCOS 7 → 1 + flip_Bt_Ip
  psi_axis      bouquet=  3.482758e-01  omfit=  3.482758e-01  rel=0.00e+00  [PASS]
  psi_boundary  bouquet=  4.402488e-02  omfit=  4.402488e-02  rel=0.00e+00  [PASS]
  B_center      bouquet= -2.024348e+00  omfit= -2.024348e+00  rel=0.00e+00  [PASS]
  Ip            bouquet= -1.200040e+06  omfit= -1.200040e+06  rel=0.00e+00  [PASS]
  FPOL          max|diff|/max|val| = 0.00e+00  [PASS]
  PRES          max|diff|/max|val| = 0.00e+00  [PASS]
  PPRIME        max|diff|/max|val| = 0.00e+00  [PASS]
  FFPRIM        max|diff|/max|val| = 0.00e+00  [PASS]
  QPSI          max|diff|/max|val| = 0.00e+00  [PASS]
  PSIRZ         max|diff|/max|val| = 0.00e+00  [PASS]
  ————————————————————————————————————————
  Overall: ALL PASS


## Summary

In [10]:
results = []
results.append(('COCOS 1 → 7',  run_cocosify_comparison(1, 7, 'COCOS 1 → 7')))
results.append(('COCOS 1 → 11', run_cocosify_comparison(1, 11, 'COCOS 1 → 11')))
results.append(('COCOS 7 → 1',  run_cocosify_comparison(7, 1, 'COCOS 7 → 1')))
results.append(('COCOS 7 → 11', run_cocosify_comparison(7, 11, 'COCOS 7 → 11')))

# flip test
beq = read_geqdsk(GFILE, cocos=7)
beq.cocosify(1)
beq.flip_Bt_Ip()
oeq = OMFITgeqdsk(GFILE)
oeq.load(raw=True)
oeq._cocos = 7
oeq.cocosify(1, True, True)
oeq.flip_Bt_Ip()
results.append(('7→1 + flip', compare(beq._raw, oeq, '7→1 + flip')))

print('\n' + '=' * 40)
print('SUMMARY')
print('=' * 40)
for name, ok in results:
    print(f'  {name:20s}  {"PASS" if ok else "FAIL"}')

Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...

  COCOS 1 → 7
  psi_axis      bouquet= -3.482758e-01  omfit= -3.482758e-01  rel=0.00e+00  [PASS]
  psi_boundary  bouquet= -4.402488e-02  omfit= -4.402488e-02  rel=0.00e+00  [PASS]
  B_center      bouquet=  2.024348e+00  omfit=  2.024348e+00  rel=0.00e+00  [PASS]
  Ip            bouquet=  1.200040e+06  omfit=  1.200040e+06  rel=0.00e+00  [PASS]
  FPOL          max|diff|/max|val| = 0.00e+00  [PASS]
  PRES          max|diff|/max|val| = 0.00e+00  [PASS]
  PPRIME        max|diff|/max|val| = 0.00e+00  [PASS]
  FFPRIM        max|diff|/max|val| = 0.00e+00  [PASS]
  QPSI          max|diff|/max|val| = 0.00e+00  [PASS]
  PSIRZ         max|diff|/max|val| = 0.00e+00  [PASS]
  ————————————————————————————————————————
  Overall: ALL PASS
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...
Flux surfaces from 257x257 gEQDSK
Levels based on psi ...

  COCOS 1 → 11
  psi_axis      bo